# PT-REWARD-MODEL. Reward Model Training, Calibration, and Overoptimization | 奖励模型训练、校准与过度优化

**难度：** Hard | **环境：** CPU-first | **标签：** `后训练`, `奖励模型`, `偏好学习` | **目标人群：** 奖励优化机制学习者

---

## 本节导读

奖励模型把回答压缩成标量分数，但分数变大不等于真实质量变好。偏好对的可比性、分数差的尺度和隐藏评测上的一致性，都会影响奖励能否安全地进入策略更新。

本节沿着“偏好对 → 标量评分 → 成对目标 → 留出校准 → 过度优化审计”建立完整机制。完成后，你应能计算成对奖励损失、选择校准温度，并识别代理奖励继续上升而真实质量已经下降的候选。

**关键词：** `reward model`, `pairwise loss`, `calibration`, `overoptimization`

## 前置阅读

**导语：** 先理解偏好样本的同提示可比性，以及奖励、价值和优势承担的不同职责。

- [Part 02 · 50 偏好数据与对齐评测](./50_Preference_Data_and_Evaluation.ipynb)
- [Part 02 · PT-REWARD-SIGNAL 奖励、价值与优势信号](./PT-REWARD-SIGNAL_Reward_Model_Value_and_Advantage.ipynb)

### Step 1：偏好监督怎样变成可用的奖励信号

同一提示下的优选回答与非优选回答先经过同一个奖励模型，得到两个标量分数。训练目标扩大优选分数与非优选分数之间的差值；留出偏好对随后检查排序正确率和分数尺度。候选策略产生的新回答还要经过隐藏质量或规则验证，避免只提高奖励模型分数。

| 阶段 | 输入 | 形成的证据 |
|:---|:---|:---|
| 偏好监督 | 同提示的优选与非优选回答 | 可比较的成对标签 |
| 奖励模型训练 | 两个标量分数及其差值 | 成对损失与排序方向 |
| 留出校准 | 未参与训练的偏好对 | 排序准确率、置信度与尺度 |
| 策略候选审计 | 新策略回答、代理奖励与隐藏质量 | 奖励提升是否仍对应真实改进 |

![奖励模型从偏好监督到策略候选审计](../docs/public/02_PyTorch_Algorithms/pt_reward_model_overview.svg)

### Step 2：成对目标只约束相对排序

Bradley–Terry 成对模型把分数差映射成“优选回答胜出”的概率。分数差为零时概率为二分之一；差值为正且持续增大时，成对损失下降。该目标约束同一提示下的相对顺序，不保证不同提示之间的绝对分数可以直接比较。

| 量 | 含义 | 需要检查的性质 |
|:---|:---|:---|
| 优选分数与非优选分数 | 奖励模型对同一提示下两个回答的标量输出 | 两者必须来自相同模型快照 |
| 分数差 | 优选分数减去非优选分数 | 正值表示排序方向正确 |
| 成对概率 | 分数差经过逻辑函数后的胜出概率 | 只表达当前偏好对内的相对置信度 |
| 成对损失 | 正确偏好方向的负对数似然 | 错序和小边际应受到更大惩罚 |

### Step 3：校准与隐藏质量共同限制奖励过度优化

校准温度只调整分数差映射到概率时的尺度，不改变排序方向。它应在独立留出对上选择，不能用训练集损失替代。策略优化阶段还要同时观察代理奖励与隐藏质量：当最高奖励候选低于固定质量门槛，或奖励排序与隐藏质量明显背离时，应停止继续追逐代理分数。

| 现象 | 可能原因 | 决策 |
|:---|:---|:---|
| 排序正确但置信度过高 | 分数尺度过大 | 调整校准温度，不重写偏好标签 |
| 训练准确率高、留出准确率低 | 偏好过拟合或数据泄漏 | 拒绝当前奖励模型并修复数据切分 |
| 代理奖励上升、隐藏质量下降 | 奖励漏洞或分布外策略行为 | 停止策略更新，补充失败样本与约束 |
| 奖励和隐藏质量同步改善 | 代理信号仍有效 | 保留候选并进入受约束策略更新 |

![奖励校准与过度优化的双重门槛](../docs/public/02_PyTorch_Algorithms/pt_reward_model_calibration.svg)

### Step 4：实现成对训练、留出校准与过度优化审计

本题用成对分数、留出标签和候选策略的两类指标复现奖励模型的三个关键判断。骨架提供输入校验、候选温度遍历和报告结构；你需要补全成对目标、校准候选选择和过度优化判定。测试覆盖正确排序、错序、温度选择、质量退化和非法输入。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|:---|:---|:---|:---|
| TODO 1：计算成对奖励目标<br>`pairwise_reward_objective` | 从优选与非优选分数得到分数差、损失和排序准确率 | 两组分数同形；正边际降低损失 | `test_pairwise_reward_objective`、`test_reward_input_contract` |
| TODO 2：选择留出校准温度<br>`select_reward_temperature` | 用留出偏好标签选择 Brier 误差最低的温度 | 温度为正；并列时选择更小温度 | `test_temperature_calibration`、`test_invalid_temperature_candidates` |
| TODO 3：审计奖励过度优化<br>`audit_reward_overoptimization` | 比较最高奖励候选的隐藏质量与固定门槛 | 不用代理奖励替代隐藏质量；保留候选身份 | `test_overoptimization_gate`、`test_aligned_reward_candidate` |

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def pairwise_reward_objective(chosen_scores, rejected_scores):
    """计算同提示偏好对的 Bradley–Terry 目标。"""
    chosen_scores = torch.as_tensor(chosen_scores, dtype=torch.float32)
    rejected_scores = torch.as_tensor(rejected_scores, dtype=torch.float32)
    if chosen_scores.shape != rejected_scores.shape or chosen_scores.numel() == 0:
        raise ValueError("chosen_scores 与 rejected_scores 必须同形且非空")

    # ==========================================
    # TODO 1: 计算成对奖励目标
    # ==========================================
    # score_margin = ???  # [pairs]；正值表示优选回答得分更高
    # pair_losses = ???  # [pairs]；正确边际增大时损失应下降
    # accuracy = ???  # 标量；严格正边际所占比例
    raise NotImplementedError("TODO 1: 请计算成对奖励目标")
    return {
        "loss": float(pair_losses.mean().item()),
        "accuracy": float(accuracy.item()),
        "margins": score_margin,
    }


def select_reward_temperature(margins, labels, candidate_temperatures):
    """在留出偏好对上选择概率校准温度。"""
    margins = torch.as_tensor(margins, dtype=torch.float32)
    labels = torch.as_tensor(labels, dtype=torch.float32)
    temperatures = [float(value) for value in candidate_temperatures]
    if margins.shape != labels.shape or margins.numel() == 0:
        raise ValueError("margins 与 labels 必须同形且非空")
    if not temperatures or any(value <= 0 for value in temperatures):
        raise ValueError("候选温度必须为正")

    records = []
    for temperature in temperatures:
        probabilities = torch.sigmoid(margins / temperature)
        brier = torch.mean((probabilities - labels) ** 2).item()
        records.append({"temperature": temperature, "brier": float(brier)})

    # ==========================================
    # TODO 2: 选择留出校准温度
    # ==========================================
    # best_record = ???  # Brier 误差最低；并列时温度更小
    raise NotImplementedError("TODO 2: 请选择留出校准温度")
    return {"selected_temperature": best_record["temperature"], "records": records}


def audit_reward_overoptimization(candidate_ids, reward_scores, quality_scores, min_quality):
    """检查最高代理奖励候选是否仍满足隐藏质量门槛。"""
    if not (len(candidate_ids) == len(reward_scores) == len(quality_scores)) or not candidate_ids:
        raise ValueError("候选身份、奖励和质量必须等长且非空")
    rewards = torch.as_tensor(reward_scores, dtype=torch.float32)
    qualities = torch.as_tensor(quality_scores, dtype=torch.float32)

    # ==========================================
    # TODO 3: 审计奖励过度优化
    # ==========================================
    # selected_index = ???  # 代理奖励最高候选的位置
    # selected_quality = ???  # 该候选的隐藏质量
    # decision = ???  # 质量达到门槛时 accept，否则 reject_overoptimization
    raise NotImplementedError("TODO 3: 请审计奖励过度优化")
    return {
        "candidate_id": candidate_ids[selected_index],
        "reward": float(rewards[selected_index].item()),
        "quality": float(selected_quality),
        "decision": decision,
    }

### 测试

测试分别验证成对目标的方向性、留出温度的选择规则，以及代理奖励与隐藏质量发生冲突时的拒绝条件。

In [ ]:
def test_pairwise_reward_objective():
    """正确排序应得到有限损失、满准确率和正边际。"""
    report = pairwise_reward_objective([2.0, 1.2], [0.5, -0.2])
    assert report["accuracy"] == 1.0
    assert torch.all(report["margins"] > 0)
    assert 0.0 < report["loss"] < 1.0


def test_reward_input_contract():
    """错序应降低准确率，形状不一致必须被拒绝。"""
    report = pairwise_reward_objective([0.0, 2.0], [1.0, 1.0])
    assert report["accuracy"] == 0.5
    try:
        pairwise_reward_objective([1.0], [0.0, 1.0])
    except ValueError:
        pass
    else:
        raise AssertionError("形状不一致必须抛出 ValueError")


def test_temperature_calibration():
    """留出标签应选择 Brier 误差最低的候选温度。"""
    result = select_reward_temperature([0.2, 0.4, -0.3, -0.8], [1, 1, 0, 0], [0.25, 0.5, 1.0, 2.0])
    assert result["selected_temperature"] == min(
        result["records"], key=lambda item: (item["brier"], item["temperature"])
    )["temperature"]


def test_invalid_temperature_candidates():
    """空候选或非正温度不能进入校准比较。"""
    for candidates in ([], [0.0, 1.0], [-1.0]):
        try:
            select_reward_temperature([0.2], [1.0], candidates)
        except ValueError:
            pass
        else:
            raise AssertionError("非法温度候选必须抛出 ValueError")


def test_overoptimization_gate():
    """最高代理奖励候选质量退化时必须拒绝。"""
    report = audit_reward_overoptimization(
        ["baseline", "candidate_a", "candidate_b"],
        [0.2, 0.8, 1.4],
        [0.82, 0.84, 0.61],
        min_quality=0.80,
    )
    assert report["candidate_id"] == "candidate_b"
    assert report["decision"] == "reject_overoptimization"


def test_aligned_reward_candidate():
    """最高代理奖励与隐藏质量同时过门槛时可以准入。"""
    report = audit_reward_overoptimization(
        ["baseline", "candidate"], [0.2, 0.9], [0.82, 0.86], min_quality=0.80
    )
    assert report["candidate_id"] == "candidate"
    assert report["decision"] == "accept"


for test_fn in (
    test_pairwise_reward_objective,
    test_reward_input_contract,
    test_temperature_calibration,
    test_invalid_temperature_candidates,
    test_overoptimization_gate,
    test_aligned_reward_candidate,
):
    test_fn()
print("All reward model mechanism tests passed.")

---

🛑 **STOP HERE** 🛑

> 请先完成题目区并运行测试，再查看参考实现。

---

## 参考代码与解析

### 代码

In [ ]:
def pairwise_reward_objective(chosen_scores, rejected_scores):
    """计算同提示偏好对的 Bradley–Terry 目标。"""
    chosen_scores = torch.as_tensor(chosen_scores, dtype=torch.float32)
    rejected_scores = torch.as_tensor(rejected_scores, dtype=torch.float32)
    if chosen_scores.shape != rejected_scores.shape or chosen_scores.numel() == 0:
        raise ValueError("chosen_scores 与 rejected_scores 必须同形且非空")

    # TODO 1：计算成对奖励目标
    score_margin = chosen_scores - rejected_scores
    pair_losses = F.softplus(-score_margin)
    accuracy = (score_margin > 0).float().mean()
    return {
        "loss": float(pair_losses.mean().item()),
        "accuracy": float(accuracy.item()),
        "margins": score_margin,
    }


def select_reward_temperature(margins, labels, candidate_temperatures):
    """在留出偏好对上选择概率校准温度。"""
    margins = torch.as_tensor(margins, dtype=torch.float32)
    labels = torch.as_tensor(labels, dtype=torch.float32)
    temperatures = [float(value) for value in candidate_temperatures]
    if margins.shape != labels.shape or margins.numel() == 0:
        raise ValueError("margins 与 labels 必须同形且非空")
    if not temperatures or any(value <= 0 for value in temperatures):
        raise ValueError("候选温度必须为正")

    records = []
    for temperature in temperatures:
        probabilities = torch.sigmoid(margins / temperature)
        brier = torch.mean((probabilities - labels) ** 2).item()
        records.append({"temperature": temperature, "brier": float(brier)})

    # TODO 2：选择留出校准温度
    best_record = min(records, key=lambda item: (item["brier"], item["temperature"]))
    return {"selected_temperature": best_record["temperature"], "records": records}


def audit_reward_overoptimization(candidate_ids, reward_scores, quality_scores, min_quality):
    """检查最高代理奖励候选是否仍满足隐藏质量门槛。"""
    if not (len(candidate_ids) == len(reward_scores) == len(quality_scores)) or not candidate_ids:
        raise ValueError("候选身份、奖励和质量必须等长且非空")
    rewards = torch.as_tensor(reward_scores, dtype=torch.float32)
    qualities = torch.as_tensor(quality_scores, dtype=torch.float32)

    # TODO 3：审计奖励过度优化
    selected_index = int(torch.argmax(rewards).item())
    selected_quality = float(qualities[selected_index].item())
    decision = "accept" if selected_quality >= float(min_quality) else "reject_overoptimization"
    return {
        "candidate_id": candidate_ids[selected_index],
        "reward": float(rewards[selected_index].item()),
        "quality": selected_quality,
        "decision": decision,
    }

### 解析

#### TODO 1：计算成对奖励目标

- 成对监督只关心同一提示下的分数差；优选分数高于非优选分数时，边际为正且损失随边际增大而下降。
- 答案用 `softplus` 稳定计算负对数似然，测试同时检查正确排序、错序和输入形状。

#### TODO 2：选择留出校准温度

- 温度改变概率置信度但不改变分数排序，因此必须在未参与训练的偏好对上比较校准误差。
- 答案按 Brier 误差选择候选，并用温度作为并列规则；测试核对最优记录并拒绝非法候选。

#### TODO 3：审计奖励过度优化

- 策略选择不能只看代理奖励；最高奖励候选仍必须通过独立隐藏质量门槛。
- 答案先定位最高奖励候选，再依据其隐藏质量形成决策；测试覆盖奖励与质量冲突以及同步改善两种情况。

## 相关阅读

- [Part 02 · 16 PPO / RLHF](./16_RLHF_PPO_Memory.ipynb)
- [Part 02 · 18 GRPO 与可验证奖励](./18_GRPO_Loss_Tutorial.ipynb)
- [Part 02 · PT-REWARD-SIGNAL 奖励、价值与优势信号](./PT-REWARD-SIGNAL_Reward_Model_Value_and_Advantage.ipynb)